In [1]:
import os
os.environ["GOOGLE_API_KEY"] = 'AQUzQ'

In [2]:
!pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into acc

In [3]:
!pip install langchain_core requests

In [4]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [5]:
#tool create

@tool
def multiply(a:int, b:int)-> int:
  """Given two numbers a and b , this tool returns their product"""
  return a*b

In [6]:
print(multiply.invoke({'a':3,'b':4}))

12


In [7]:
multiply.description

'Given two numbers a and b , this tool returns their product'

In [8]:
#tool binding

llm = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash'
)

In [9]:
llm_with_tools = llm.bind_tools([multiply])

In [10]:
query = HumanMessage("what's 8 multiplied with 7?")

In [11]:
messages = [query]

In [12]:
messages

[HumanMessage(content="what's 8 multiplied with 7?", additional_kwargs={}, response_metadata={})]

In [13]:
result = llm_with_tools.invoke(messages)

In [14]:
messages.append(result)

In [15]:
result.tool_calls[0]['args']

{'a': 8, 'b': 7}

In [16]:
multiply.invoke(result.tool_calls[0]['args'])

56

In [17]:
tool_result = multiply.invoke(result.tool_calls[0])

In [18]:
messages.append(tool_result)

In [19]:
llm_with_tools.invoke(messages).content

[{'type': 'text',
  'text': '8 multiplied by 7 is 56.',
  'extras': {'signature': 'Eq0BCqoBARFNMg87uLJit+N/Pkz+dslXNIxVE+wBZgJfdyT4oZyA7J9VWSgRUPRKEVmWKzjdYwgJEcQEG8nr976A5pZsYqMY2DXi3rx+fhp/uvkUe3UtHQjh6yySjgalBhTlefvxtRn09MquhkcUuVaJC0+COS0C9eusT+dIktEOaPULTmF1l90gPcbFA4ssjzEcl7B6eVXXmItX9oZoo9bgGQjBFiilDhW8vxZelNE='}}]

In [20]:
messages

[HumanMessage(content="what's 8 multiplied with 7?", additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 8, "b": 7}'}, '__gemini_function_call_thought_signatures__': {'call_298415': 'EpsCCpgCARFNMg//ZwHoLkB6muZBp5psdLmtuBoHzbd+mui66kSjw3E5pzgi1HOpt834UOJyfUTnGlb+jNQDyxrvzxRjBlJCIY0POT8+E9sW2FZMonScBVoO9P57QXsJmVU1MfQtMhHWEa9Fq3jcUmWhxvlNi7bpV60AVfyxVieFheiA9KgUD1GwosFT1mB85R8NruKRFr3tTkHdRjJmwaJA1S11JxEo/fOA6BL2ZPkBvNaJGwdP3AWN6CfHdMo4XL92fuVsz/OKIwmmg/7CRR59nOHd/+I/4KPgmA3gMXXJIhpozgtOjAmz0o435MXUmMMSZzKvEkhwRpenbyNlzlpx1FH44WVlDzfMxq62oH2IMNpFtJ0i4/XbX2PK6w=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a06ffd-d4b8-7751-a0ed-9d949ec3f900-0', tool_calls=[{'name': 'multiply', 'args': {'a': 8, 'b': 7}, 'id': 'call_298415', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

Currency conversion tool

In [47]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches the currency conversion factor between a given base currency and a target currency"""

    url = f'https://v6.exchangerate-api.com//pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()

@tool
def convert_currency(base_currency: float, conversion_factor: Annotated[float,InjectedToolArg]) -> float:
    """This function converts an amount from one currency to another"""
    return base_currency * conversion_factor

In [48]:
get_conversion_factor.invoke({
    'base_currency': 'USD',
    'target_currency': 'INR'
})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1788566402,
 'time_last_update_utc': 'Sat, 05 Sep 2026 00:00:02 +0000',
 'time_next_update_unix': 1788652802,
 'time_next_update_utc': 'Sun, 06 Sep 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.5176}

In [22]:
convert_currency.invoke({'base_currency':1000, 'conversion_factor':94.5176})

94517.6

In [49]:
#tool binding
llm = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash'
)


In [50]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert_currency])

In [51]:
messages = [HumanMessage('What is the conversion factor between GBP and INR and based on that can you convert 10 GBP to INR')]

In [52]:
messages

[HumanMessage(content='What is the conversion factor between GBP and INR and based on that can you convert 10 GBP to INR', additional_kwargs={}, response_metadata={})]

In [53]:
ai_message = llm_with_tools.invoke(messages)

In [54]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'GBP'},
  'id': 'call_659113',
  'type': 'tool_call'}]

In [55]:
messages

[HumanMessage(content='What is the conversion factor between GBP and INR and based on that can you convert 10 GBP to INR', additional_kwargs={}, response_metadata={})]

In [56]:
ai_message

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "GBP"}'}, '__gemini_function_call_thought_signatures__': {'call_659113': 'EvUICvIIARFNMg/aJHcDl41R9ZZT2+WLC47tRdQKuD21eYFyH3mQt67k1omPU3JK/zip+9sztx/M+tM+EcmM67hlLScqvH0Ki3PdR8QGLEVcOhhpFmJCD9NFskLDzVx6eOGoVZpJ7tA1vsr9JedJdqS3Cm9tr1ixekpMO/0b+zXr0QAnB9BTYsZCi1iWNCx20YeXNpAZuGAAhu/TyRszXK0WHijg91uLaCr2DzciHeIpbEcIdwTmqJslTRW4cDPXBDrIwlQy3q/pP04DoTzZF8mj93d/jIUJtgcpQ6JQgXMYOyaGzhBtEXjmrYMe6CNGv+KlKfRlcHvm/X/AinMmML3bQK6BoA3KLpmxvB8O8LPOJa+Ud3geYqTkSJI+If0fUoihuD3gTMvo5hlQG5ailDSD8yDPe7WeWU/O0OovqfBii/q71pwZgISYUoKOFuNFpQTfPBVP0KUALuS4QIQeKvydzxblakkCyDF4BSCRB1f/kQDuqUtt3kkdX9EbmTbOKkPdwMvxnJDaeUQvLB+O/DxT/U2PGunlj8S3DitWbpypsj4SC47SWPRG+x6KDSkFAvzJlqRI1wXt1+O1niKSjtLC/UcYpYU1HyZ7RXD3xqChkjV8Qu1DjKvdnQZtg3SUyQu/SsdKSbFdhgjDJbTlX032vZWuVGNEODt/zXW9uhVndjwh4p+Q0U37Pgde9gVWAfQrSrejI3XYn484S6eSRiYciBKuf7sSDXaBOhxw8jcidVuzYxsgRiHHNe9/Um5AHypJvPBv

In [57]:
messages.append(ai_message)

In [58]:
import json
for tool_call in ai_message.tool_calls:
  #execute the 1st tool and get the value of conversion rate
  if tool_call['name']== 'get_conversion_factor':
    tool_message1= get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']

    messages.append(tool_message1)

    ai_message = llm_with_tools.invoke(messages)
    messages.append(ai_message)

    if tool_call['name'] == 'convert_currency':
      tool_message2 = convert_currency.invoke({
          'base_currency': 10,
          'conversion_factor': conversion_rate
      })

      messages.append(tool_message2)

In [59]:
messages

[HumanMessage(content='What is the conversion factor between GBP and INR and based on that can you convert 10 GBP to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "GBP"}'}, '__gemini_function_call_thought_signatures__': {'call_659113': 'EvUICvIIARFNMg/aJHcDl41R9ZZT2+WLC47tRdQKuD21eYFyH3mQt67k1omPU3JK/zip+9sztx/M+tM+EcmM67hlLScqvH0Ki3PdR8QGLEVcOhhpFmJCD9NFskLDzVx6eOGoVZpJ7tA1vsr9JedJdqS3Cm9tr1ixekpMO/0b+zXr0QAnB9BTYsZCi1iWNCx20YeXNpAZuGAAhu/TyRszXK0WHijg91uLaCr2DzciHeIpbEcIdwTmqJslTRW4cDPXBDrIwlQy3q/pP04DoTzZF8mj93d/jIUJtgcpQ6JQgXMYOyaGzhBtEXjmrYMe6CNGv+KlKfRlcHvm/X/AinMmML3bQK6BoA3KLpmxvB8O8LPOJa+Ud3geYqTkSJI+If0fUoihuD3gTMvo5hlQG5ailDSD8yDPe7WeWU/O0OovqfBii/q71pwZgISYUoKOFuNFpQTfPBVP0KUALuS4QIQeKvydzxblakkCyDF4BSCRB1f/kQDuqUtt3kkdX9EbmTbOKkPdwMvxnJDaeUQvLB+O/DxT/U2PGunlj8S3DitWbpypsj4SC47SWPRG+x6KDSkFAvzJlqRI1wXt1+O1niKSjtLC/UcYpYU

In [61]:
llm_with_tools.invoke(messages).content

[{'type': 'text',
  'text': 'The conversion factor from **GBP** to **INR** is **127.7407**.\n\nBased on this exchange rate:\n$$\\text{10 GBP} = 10 \\times 127.7407 = \\text{\\textbf{1,277.41 INR}}$$',
  'extras': {'signature': 'EoYGCoMGARFNMg+SQDHtLP0MNN7/It9JDaD6IsBzzN8lawKSNCQa2MplMJCyZwL9gzSoHsJF562zpJ2hyEZOHVfDOPnXObyFRbmZYbW6wOVKpkHCjsTRBlin+06Mw+5JuzqUs6jlt4GsTr73L0BcPOBRyawLTZHFMBvCBucMwjLbzqUtpMEQfl2oEl8NGFd0b17/mujze9KypgPf1gb/C7IJ0gVeaeKWwEjcRlMauuNpK1+D0DL/FI9QF57gELpIZzL0wfUvXVkwRP2lJPkH0JQB33nYv3oSuukpOmOyWom5TTqtwCSsDKMp5BJzqQqe9F1DgN9phIzw7iofEjnRsZtCAME/UDs9cLk2Fjc9QSMhZSL7qNadzHdMA1TK99lqTNvIAOGHnYOWRF52eY7IYCuEjJZyroGo2Ol8kPZZSiEax9NhqEl42gnUgAb2bL8uab1tqPA1QrDZsa6XMOrsTvbb7RkkQ2XIyaFVqVd5e4cWh1BzH2/nAIMGASyfcJ/15WW94F3V3Mxil1hrNpGEpCkeyA13euIBhzrn/FGkidQKAmE1yz1toF57etAusZV0yGsoeJFMjwNK9AHUOclDKROc138q3obQ1qtNudLpI1E4ZPzRT6bmcOq0R7kcPe4CTvZjmD7G4X/M0OAxWv7K9o9W2WDtuR050ZAsRTTWoBJi0bdVb/DUXP1SxibuWHfSMCq5xsSoeU4TgUHb5a8m8LGPzriDuHkqZ6NKvBNSsau2XUD455mwTwVbOaGzAtRc+mFM